## 1. Bibliotecas

In [ ]:
import json
import os, sys, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

## 2. Configuração do ambiente

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/httpsemilly/xplica.ai.git'
REPO_DIR = Path('/content/xplica-ai')

if IN_COLAB:
  if not REPO_DIR.exists():
    !git clone --filter=blob:none --sparse {REPO_URL} {REPO_DIR}
    !git -C {REPO_DIR} sparse-checkout set data

  os.chdir(REPO_DIR)

  !pip -q install modal lime statsmodels python-dotenv
  !python -m spacy download pt_core_news_sm

  from google.colab import drive
  drive.mount('/content/drive', force_remount=False)

  DRIVE_ROOT = Path('/content/drive/MyDrive/xplica')
  DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

  for sub in ('models', 'results'):
    local = REPO_DIR / sub
    drive_dir = DRIVE_ROOT / sub
    drive_dir.mkdir(parents=True, exist_ok=True)

    if not local.exists():
        local.symlink_to(drive_dir)
    print(f'{local} -> {drive_dir}')

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').exists()), None)

if ROOT is None:
  raise RuntimeError("A pasta 'data/' não foi encontrada. Clone o repo e rode a partir da raiz.")

if str(ROOT) not in sys.path:
  sys.path.insert(0, str(ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'ROOT = {ROOT}')
print(f'device = {device}')

if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)}')
elif IN_COLAB:
  print("Sem GPU. Ambiente de execução -> Alterar tipo -> T4 GPU.")

In [ ]:
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
MAX_LENGTH = 512
label_map = {'fake': 0, 'true': 1}

MODELS_DIR = ROOT / 'models'
HF_USER = os.environ.get('XPLICA_HF_USER')
PUSH_TO_HUB = os.environ.get('XPLICA_PUSH_TO_HUB') == '1'

## 3. Carregamento e preparação dos dados

In [ ]:
df_news = pd.read_csv(ROOT / 'data' / 'processed' / 'news_dataset.csv')
print(f'Total news loaded: {len(df_news)}')

## 4. Funções auxiliares

In [ ]:
class NewsDataset(torch.utils.data.Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

In [ ]:
def tokenize_texts(texts, tokenizer, max_length=MAX_LENGTH):
  return tokenizer(list(texts), padding='max_length', truncation=True, max_length=max_length, return_tensors='pt')

In [ ]:
def prepare_dataset(df, tokenizer):
  return NewsDataset(tokenize_texts(df['text'], tokenizer), df['label_id'].tolist())

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
  cm = confusion_matrix(labels, predictions)

  return {
    'accuracy': accuracy_score(labels, predictions),
    'precision': precision, 'recall': recall, 'f1': f1,
    'tn': cm[0][0], 'fp': cm[0][1], 'fn': cm[1][0], 'tp': cm[1][1]
  }

In [ ]:
def filter_test_data(df, fake_origin, test_split='test'):
  df_true = df[(df['label'] == 'true') & (df['split'] == test_split)].copy()
  df_fake = df[(df['label'] == 'fake') & (df['origin'] == fake_origin) & (df['split'] == test_split)].copy()
  df_test = pd.concat([df_true, df_fake]).copy()
  df_test['label_id'] = df_test['label'].map(label_map)

  return df_test

In [ ]:
def filter_train_data(df, train_origins, train_split='train', synthetic_ratio=None, seed=42):
  if len(train_origins) == 1:
    df_train = df[(df['origin'].isin(train_origins)) & (df['split'] == train_split)].copy()
  else:
    df_true = df[(df['label'] == 'true') & (df['split'] == train_split)].copy()
    df_fake_human = df[(df['label'] == 'fake') & (df['origin'] == 'human') & (df['split'] == train_split)]
    df_fake_synthetic = df[(df['label'] == 'fake') & (df['origin'] == 'synthetic') & (df['split'] == train_split)]

    n_synthetic = int(len(df_fake_human) * synthetic_ratio)
    n_human = len(df_fake_human) - n_synthetic

    df_train = pd.concat([
      df_true,
      df_fake_human.sample(n=n_human, random_state=seed),
      df_fake_synthetic.sample(n=n_synthetic, random_state=seed)
    ]).copy()

  df_train['label_id'] = df_train['label'].map(label_map)

  return df_train

## 5. Definição dos cenários

No treinamento misto, considerar 50% de fake news sintéticas e 50% de fake news humanas

| Cenário                 | Treinamento | Teste     | Objetivo                                                                                    |
| ----------------------- | ----------- | --------- | ------------------------------------------------------------------------------------------- |
| **1. Baseline**         | Humano      | Humano    | Estabelecer o desempenho de referência                                                      |
| **2. Problema**         | Humano      | Sintético | Verificar a generalização para textos sintéticos                                            |
| **3. Solução**          | Misto       | Sintético | Avaliar se a inclusão de dados sintéticos no treinamento melhora o desempenho nesse domínio |
| **4. Efeito colateral** | Misto       | Humano    | Verificar o impacto do treinamento misto sobre textos humanos                               |


In [ ]:
scenarios = {
  'scenario1': {
    'train_origins': ['human'],
    'test_origin': 'human',
    'train_model': True
  },
  'scenario2': {
    'train_origins': ['human'],
    'test_origin': 'synthetic',
    'train_model': False,
    'base_model': 'scenario1'
  },
  'scenario3': {
    'train_origins': ['human', 'synthetic'],
    'test_origin': 'synthetic',
    'train_model': True,
    'synthetic_ratio': 0.5
  },
  'scenario4': {
    'train_origins': ['human', 'synthetic'],
    'test_origin': 'human',
    'train_model': False,
    'base_model': 'scenario3'
  }
}

## 6. Execução dos cenários

In [ ]:
def hub_repo(name):
  if not HF_USER:
    raise RuntimeError('Set the XPLICA_HF_USER environment variable to use the Hugging Face Hub.')

  return f'{HF_USER}/xplica-bertimbau-{name}'

In [ ]:
def save_scenario_model(model, name):
  path = MODELS_DIR / f'bertimbau-{name}'
  model.save_pretrained(path)
  assert (path / 'config.json').exists(), 'save failed'

  if PUSH_TO_HUB:
    model.push_to_hub(hub_repo(name), private=True)

  print(f'Model saved to {path}')

In [ ]:
def load_scenario_model(name):
  path = MODELS_DIR / f'bertimbau-{name}'
  source = str(path) if path.exists() else hub_repo(name)

  return AutoModelForSequenceClassification.from_pretrained(source)

In [ ]:
def train_scenario(name, df_train, df_test, tokenizer):
  train_dataset = prepare_dataset(df_train, tokenizer)
  test_dataset = prepare_dataset(df_test, tokenizer)

  model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

  training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / 'checkpoints' / name),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    seed=42
  )

  trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=test_dataset, compute_metrics=compute_metrics)
  trainer.train()

  return model, trainer, test_dataset

In [ ]:
def evaluate_scenario(model, df_test, tokenizer):
  test_dataset = prepare_dataset(df_test, tokenizer)
  trainer = Trainer(model=model, compute_metrics=compute_metrics)

  return trainer.evaluate(test_dataset)

In [ ]:
results = {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)

In [ ]:
cfg = scenarios['scenario1']
df_train_s1 = filter_train_data(df_news, train_origins=cfg['train_origins'])
df_test_s1 = filter_test_data(df_news, fake_origin=cfg['test_origin'])

model_s1, trainer_s1, test_dataset_s1 = train_scenario('scenario1', df_train_s1, df_test_s1, tokenizer)
save_scenario_model(model_s1, 'scenario1')

results['scenario1'] = trainer_s1.evaluate(test_dataset_s1)
print(results['scenario1'])

In [ ]:
cfg = scenarios['scenario2']
df_test_s2 = filter_test_data(df_news, fake_origin=cfg['test_origin'])

model_s2 = load_scenario_model(cfg['base_model'])
results['scenario2'] = evaluate_scenario(model_s2, df_test_s2, tokenizer)
print(results['scenario2'])

In [ ]:
cfg = scenarios['scenario3']
df_train_s3 = filter_train_data(df_news, train_origins=cfg['train_origins'], synthetic_ratio=cfg['synthetic_ratio'])
df_test_s3 = filter_test_data(df_news, fake_origin=cfg['test_origin'])

model_s3, trainer_s3, test_dataset_s3 = train_scenario('scenario3', df_train_s3, df_test_s3, tokenizer)
save_scenario_model(model_s3, 'scenario3')

results['scenario3'] = trainer_s3.evaluate(test_dataset_s3)
print(results['scenario3'])

In [ ]:
cfg = scenarios['scenario4']
df_test_s4 = filter_test_data(df_news, fake_origin=cfg['test_origin'])

model_s4 = load_scenario_model(cfg['base_model'])
results['scenario4'] = evaluate_scenario(model_s4, df_test_s4, tokenizer)
print(results['scenario4'])

In [ ]:
CKPT_ROOT = MODELS_DIR / 'checkpoints'
FINAL_DIR = MODELS_DIR

def recover_best_model(name):
  src = CKPT_ROOT / name
  dst = FINAL_DIR / f'bertimbau-{name}'

  if (dst / 'config.json').exists() and (dst / 'model.safetensors').exists():
    print(f'{name}: já consolidado em {dst}')
    return dst

  ckpts = sorted(src.glob('checkpoint-*'), key=lambda c: int(c.name.split('-')[1]))

  if not ckpts:
    print(f'{name}: sem checkpoints em {src}, pulando')
    return None

  last = ckpts[-1]
  state = json.loads((last / 'trainer_state.json').read_text())
  best = Path(state['best_model_checkpoint'])

  if not best.exists():
    best = last

  print(f'{name}: melhor checkpoint = {best.name}')

  model = AutoModelForSequenceClassification.from_pretrained(str(best))
  model.save_pretrained(str(dst))

  AutoTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False).save_pretrained(str(dst))
  print(f'{name}: consolidado em {dst}')

  return dst

for name in ('scenario1', 'scenario3'):
  recover_best_model(name)